# 07 · Beyond `BaseModel`: `TypeAdapter`, `@validate_call`, Pydantic Dataclasses & TypedDicts

**Module 15 – Pydantic: Data Validation & Settings** · Estimated time: 40 min

### 🎯 What you will learn
- `TypeAdapter`: validate/serialise **any** type (`list[int]`, `dict[str, Model]`, unions, `TypedDict`...)
- `@validate_call`: validate function arguments (and return values)
- `pydantic.dataclasses.dataclass`: dataclasses with validation
- Validating standard-library `@dataclass`es, `TypedDict`s and `NamedTuple`s
- Performance tips: reuse adapters, `model_validate_json`, `model_construct`

In [1]:
from dataclasses import dataclass as std_dataclass
from datetime import date
from typing import Annotated, TypedDict, NotRequired, Literal
from pydantic import BaseModel, TypeAdapter, ValidationError, validate_call, Field
from pydantic.dataclasses import dataclass as pydantic_dataclass

## 1. `TypeAdapter`

When the data is a list, a dict, a union or a plain type, you don't need to wrap it in a model:

In [2]:
ints = TypeAdapter(list[int])
print(ints.validate_python(["1", 2, "3"]))
print(ints.validate_json("[4, 5, 6]"))
try:
    ints.validate_python([1, "two"])
except ValidationError as e:
    print("❌", e.errors()[0]["loc"], e.errors()[0]["msg"])

class Product(BaseModel):
    sku: str
    price: float

catalog = TypeAdapter(dict[str, list[Product]])
data = catalog.validate_json('{"pens": [{"sku": "P1", "price": "10"}], "books": []}')
print(data)
print(catalog.dump_json(data))                    # serialise, too
print(TypeAdapter(list[Product]).json_schema()["items"]["$ref"])

[1, 2, 3]
[4, 5, 6]
❌ (1,) Input should be a valid integer, unable to parse string as an integer
{'pens': [Product(sku='P1', price=10.0)], 'books': []}
b'{"pens":[{"sku":"P1","price":10.0}],"books":[]}'
#/$defs/Product


Constrained and union types:

In [3]:
Age = TypeAdapter(Annotated[int, Field(ge=0, le=150)])
print(Age.validate_python("42"))
IdOrName = TypeAdapter(int | Literal["me"])
print(IdOrName.validate_python("me"), IdOrName.validate_python("7"))
print(TypeAdapter(date).validate_python("2026-10-20"))

42
me 7
2026-10-20


> 💡 **Create adapters once** (module level) and reuse them; building one is much slower than validating with it.

## 2. `@validate_call`: validated function arguments

In [4]:
@validate_call
def book_room(guests: Annotated[int, Field(ge=1, le=6)], check_in: date, nights: int = 1,
              room: Literal["single", "double", "suite"] = "double") -> str:
    return f"{room} room for {guests} from {check_in.isoformat()} × {nights} nights"

print(book_room("2", "2026-12-24", nights="3"))           # coerced like a model
try:
    book_room(9, "christmas", room="penthouse")
except ValidationError as e:
    for err in e.errors():
        print("❌", err["loc"], err["msg"])

double room for 2 from 2026-12-24 × 3 nights
❌ (0,) Input should be less than or equal to 6
❌ (1,) Input should be a valid date or datetime, input is too short
❌ ('room',) Input should be 'single', 'double' or 'suite'


In [5]:
@validate_call(validate_return=True)
def parse_score(raw: str) -> Annotated[int, Field(ge=0, le=100)]:
    return int(raw) * 10                                   # bug for inputs > 10

print(parse_score("7"))
try:
    parse_score("15")
except ValidationError as e:
    print("❌ bad return value:", e.errors()[0]["msg"])

70
❌ bad return value: Input should be less than or equal to 100


Use `@validate_call` at **boundaries** (CLI handlers, public library functions, plugin hooks), not on every internal function (validation has a cost).

## 3. Pydantic dataclasses

Same syntax as standard dataclasses, plus validation:

In [6]:
@pydantic_dataclass
class Point:
    x: float
    y: float = 0.0

print(Point(x="1.5"))
try:
    Point(x="left")
except ValidationError as e:
    print("❌", e.errors()[0]["msg"])

Point(x=1.5, y=0.0)
❌ Input should be a valid number, unable to parse string as a number


They're plain dataclasses (no `model_dump()`); use `TypeAdapter(Point).dump_python(p)` or `dataclasses.asdict(p)` to serialise.

## 4. Validating standard dataclasses, TypedDicts and NamedTuples

You can validate types you **already have**, without changing them:

In [7]:
@std_dataclass
class Address:
    city: str
    pincode: str

class UserDict(TypedDict):
    name: str
    age: int
    nickname: NotRequired[str]

addr = TypeAdapter(Address).validate_python({"city": "Pune", "pincode": "411001"})   # stdlib dataclass from a dict
print(addr, type(addr).__name__)
print(TypeAdapter(UserDict).validate_json('{"name": "Asha", "age": "31"}'))
try:
    TypeAdapter(UserDict).validate_python({"name": "Ravi"})
except ValidationError as e:
    print("❌", e.errors()[0]["loc"], e.errors()[0]["msg"])

Address(city='Pune', pincode='411001') Address
{'name': 'Asha', 'age': 31}
❌ ('age',) Field required


`BaseModel` fields can also be stdlib dataclasses or TypedDicts.

## 5. Performance tips

In [8]:
import json, timeit

class Order(BaseModel):
    id: int
    items: list[str]
    total: float

payload = json.dumps([{"id": i, "items": ["a", "b"], "total": i * 1.5} for i in range(1000)])
orders = TypeAdapter(list[Order])

t1 = timeit.timeit(lambda: [Order.model_validate(o) for o in json.loads(payload)], number=20)
t2 = timeit.timeit(lambda: orders.validate_json(payload), number=20)
t3 = timeit.timeit(lambda: [Order.model_construct(**o) for o in json.loads(payload)], number=20)
print(f"json.loads + model_validate in a loop : {t1 * 50:.1f} ms per 1000")
print(f"TypeAdapter.validate_json (one pass)  : {t2 * 50:.1f} ms per 1000")
print(f"model_construct (NO validation!)      : {t3 * 50:.1f} ms per 1000")

json.loads + model_validate in a loop : 0.8 ms per 1000
TypeAdapter.validate_json (one pass)  : 0.5 ms per 1000
model_construct (NO validation!)      : 1.2 ms per 1000


- ✅ Validate JSON **directly** (`model_validate_json`, `TypeAdapter.validate_json`): parsing happens in Rust in one pass
- Reuse `TypeAdapter`s; avoid re-defining models in loops
- `model_construct()` skips validation, but notice it was **not even faster** above: Pydantic's Rust validation is so quick that skipping it rarely pays off. Use it only for already-trusted data when validators are genuinely expensive, never for speed by default
- Discriminated unions are faster than plain unions
- For extreme throughput consider `msgspec`, but Pydantic is fast enough for almost everything

## 📝 Summary
- `TypeAdapter(T)` validates/serialises any type and makes JSON Schemas.
- `@validate_call` validates arguments (and optionally return values).
- Pydantic dataclasses add validation to the dataclass syntax; stdlib dataclasses/TypedDicts can be validated with adapters.
- Validate JSON directly and reuse adapters; skipping validation (`model_construct`) is rarely worth it.

## 🏋️ Practice
1. Use a `TypeAdapter(list[EmailStr])` to validate a list of email addresses from a text file.
2. Decorate a CLI command function with `@validate_call` and pass it string arguments from `sys.argv`.
3. Benchmark `model_validate_json` vs `json.loads` + `model_validate` for a list of 10,000 objects.